In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import lightkurve as lk
import astropy.units as u
import astropy.constants as const

# Introduction
Today we're going to explore TESS data for an active, flaring M dwarf. Your goal is to identify as many distinct flares as possible in a single Sector of TESS data, measure the energy of each event, and then make the Flare Frequency Distribution (FFD) to estimate the flare rate.

We'll be using a target identified as a flare star early in the TESS mission by [Günther+2020](https://ui.adsabs.harvard.edu/abs/2020AJ....159...60G/abstract). I've picked a target from their Table 2 that has a fairly high flare rate, but without a super-fast rotation period since rapid starspot modulations can make flare identification and energy calculation more difficult.

I suggest using `TIC 224225152`, but you are welcome to explore a different target.
They provide an estimate for $T_{eff}=3350 \, K$ and $R=0.421600 \, R_\odot$, which we will need to estimate the star's [Luminosity](https://en.wikipedia.org/wiki/Stefan–Boltzmann_law).

# Directions
1. Work with a partner (make friends, do great science!) Write down who your partner is in your notebook!
2. Download the TESS 2-min data for this star. Pick one Sector to measure flares from (your choice)
3. [Flatten](https://lightkurve.github.io/lightkurve/reference/api/lightkurve.LightCurve.flatten.html) the light curve, as we've done before. Explore the `window_length` parameter to best remove the starspot while not clipping the flares.
4. Identify the flare events in the flattened data. This "algorithm" could be as simple as picking start/stop times by-eye, or in finding continuous runs of points above the quiescent data. I suggest making a list of Start and Stop times!
5. What's the highest amplitude flare in your sample? Make a nice zoom-in plot of that event!
6. Calculate the "Equivalent Duration" for each flare event. I suggest using a trapazoidal sum to measure the area under the curve. These should have units of Seconds. Convert these to energies in units of erg by multiplying by the star's Luminosity.
7. With the array of flare event energies, plot the FFD.
8. Rename your notebook file to YOUR name(s), export to PDF, turn in via [Dropbox upload link](https://www.dropbox.com/request/R7aRKFr7m7YSN8z8CJ5K). (As with all assignments, nominally due in 1 week.)
   

In [ ]:
# we want the 2-min TESS light curve from the "SPOC" pipeline (i.e. the standard TESS pipeline)

lc = lk.search_lightcurve('TIC224225152', exptime=120, author='SPOC').download_all()

In [ ]:
lc.plot()
plt.show()

In [ ]:
k=0
# recall these tricks about turning the lightcurve into arrays for easier use later
time = np.array(lc[k].time.value, dtype=float)

# play with the window length! Use an odd number for good luck
flux = np.array(lc[k].flatten(window_length=301).flux.value, dtype=float)

plt.figure()
plt.plot(time, flux)
plt.show()


In [ ]:
# hopefully you have more than 1 flare in your sample!
t_start = [1376.27]
t_stop = [1376.30]

# once you have a list of flares times, you could loop through them and compute their Equiv Dur's
flare = (time > t_start[0]) & (time <=t_stop[0])

# how to use Trapezoidal Sum to measure the "Equivalent Duration"
EquivDur = np.trapz(flux[flare] - 1, # subtract 1 to center about 0
                    x=(time[flare] * 24*60*60) # convert time to Seconds for units
                   )

In [ ]:
# use astropy units and constants to do this calculation easily
L = ...

# you can force it to give you back the proper units
L = L.to('erg/s')

# convert ED to Energy
Energy = EquivDur * L

In [ ]:
# now let's compute the FFD. Recall from lecture: this is the reverse cumulative rate distribution


## FFD Steps:
1. Reverse sort the flares by energy (biggest to smallest). I like to use `np.argsort` to figure out the sort order (small to large) and reverse it, either with `np.flip` or `[::-1]`.
2. the FFD's x-array is just the reverse sorted flare energies
3. compute the cumulative number of events. This is just an integer list from 1 to N flares total (note: start counting at 1!)
4. divide the cumulative number of flares by the total duration of the light curve (usually in days), to give you a rate in # per Time
5. the FFD's y-array is this cumulative rate
6. plot the FFD (x,y). Use log-log scaling of the axes, and be sure to label axes correctly with units (flare energy and cumulative rate)

In [ ]:
# this is the total observing duration of the TESS Sector. But note we didn't actually look for flares in the mid-Sector gap...
total_duration = np.max(time) - np.min(time) 

# this estimates the total observing time as (N datapoints)*(typical delta t). Probably slightly better
total_duration = len(time) * np.median(np.diff(time))

# these lines probably won't work, but give you an idea what you need to do
ffd_x = Energy[Esort][::-1]
ffd_y = np.arange(1, len(Energy)+1) / total_duration

In [ ]:
plt.figure()
plt.plot(ffd_x, ffd_y)
plt.xscale('log')
plt.yscale('log')
